# ENVIROMENT
Create an empty folder for the project and initialize it:

```bash
mkdir llm-zoomcamp-2026-code
cd llm-zoomcamp-2026-code
uv init
```

This creates a `pyproject.toml` and a basic project structure.

Now add the dependencies we'll need:

```bash
uv add requests minsearch openai jupyter python-dotenv
```

This installs:

- `requests` - to fetch the FAQ dataset from the internet
- `minsearch` - a simple in-memory search engine for indexing and
  searching text
- `openai` - the OpenAI API client for calling the LLM
- `jupyter` - the notebook environment where we'll write and run code
- `python-dotenv` - to load API keys from a `.env` file

## Starting Jupyter

Start Jupyter:

```bash
uv run jupyter notebook
```

Create a new notebook. Throughout the course, you'll copy code from
the section notes into notebook cells.

Check that the OpenAI client works:

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

## (Optional) Auto-loading .env with dirdotenv

If you don't want to call `load_dotenv()` in every notebook, use
[dirdotenv](https://github.com/alexeygrigorev/dirdotenv).

It loads `.env` files automatically when you `cd` into a directory:

```bash
uv tool install dirdotenv
echo 'eval "$(dirdotenv hook bash)"' >> ~/.bashrc
```

Restart your terminal, and now whenever you enter the project
directory, the variables from `.env` are loaded automatically. No
`load_dotenv()` needed.


# RAG
We run free Zoomcamp courses at DataTalks.Club on data engineering,
machine learning, MLOps, and other topics. Each course has its own
FAQ document with common questions and answers.

Some of these documents have over 300 questions. Students ask us
things in Slack like "Can I still join after the course started?" or
"How do I get a certificate?" Finding those answers in the FAQ is
tedious.

We want a bot that takes all this knowledge and answers student
questions in natural language.

In this module, we'll build that system. But first, let's see why we
can't send the question straight to an LLM and call it a day.

## Plain LLMs lack our data

First, let's define a function to talk to the LLM:

In [2]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

## Retrieval plus generation

RAG stands for Retrieval-Augmented Generation. Generation is the LLM
producing text, and retrieval is search. We retrieve relevant documents
from our knowledge base and use them to augment what the LLM generates.
That search step is what gives the LLM the context it needs to answer
correctly.

What we just did was naive. I knew in advance which FAQ entry held the
answer and pasted it in by hand. What we want instead is to perform
search automatically. We take the student's question, find the most
relevant documents, and send those to the LLM.

In [ ]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

Absolutely—usually, yes.

If the course is still open for enrollment, you can often join after it has started. A few things to check:
- **Enrollment deadline**: Some courses allow late registration, others don’t.
- **Access to missed material**: You may be able to catch up with recordings, notes, or archived lessons.
- **Prerequisites**: Make sure you have any required background or access codes.
- **Instructor approval**: Some courses require permission to join late.

If you want, I can help you draft a short message to the instructor or check what to ask before enrolling.


In [4]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json() # body

That's the entire architecture. It comes down to three components.

The pieces are search, the prompt, and the LLM:

- search
- prompt
- LLM

<details>
    <summary>Click to view details the 'diagram'</summary>
    
```mermaid
flowchart TD
    U([User])

    APP[Application]

    DB[(DB)]
    DOCS[[D1 ... D5]]

    PROMPT[Build Prompt<br/>Question + Context]

    LLM[LLM]

    ANSWER([Answer])

    U -->|Question| APP

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP

    APP --> PROMPT
    PROMPT --> LLM

    LLM --> ANSWER
    ANSWER --> U
```
</details>


In [5]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

print("- total lenght for the documents: ",len(documents))

- total lenght for the documents:  1349


In [6]:
documents[1100]

{'id': '84301e35dd',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': "ConnectionError 'Connection aborted.' for --bind 127.0.0.1:5000",
 'answer': 'I was getting an error on the client side with this:\n\n**Client Side Error:**\n\n```plaintext\nFile "C:\\python\\lib\\site-packages\\urllib3\\connectionpool.py", line 703, in urlopen ...\n\nraise ConnectionError(err, request=request)\n\nrequests.exceptions.ConnectionError: (\'Connection aborted.\', RemoteDisconnected(\'Remote end closed connection without response\'))\n```\n\n**Server Side:**\n\nAn error was shown for Gunicorn, although the `waitress` command was running smoothly from the server side.\n\n**Solution:**\n\n- Use the IP address `0.0.0.0:8000` or `0.0.0.0:9696`. They are the ones which work most of the time.'}

## Indexing with minsearch
We already have the `documents` list from the previous section. Now
let's index it.

There are many search libraries you can use - [Apache Lucene][lucene], [Elasticsearch][elastic], [Solr][solr], and others. But these are somewhat heavy. For example, to run Elasticsearch, you need to start a Docker container.

[lucene]: https://github.com/apache/lucene
[elastic]: https://github.com/elastic/elasticsearch
[solr]: https://github.com/apache/solr


In [7]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
search_results = index.search(
    question,
    boost_dict={'question' : 2.0, 'section' : 0.5},  #boost : the question field is a stronger signal than them appearing in the section name.
    filter_dict={'course' : 'llm-zoomcamp'},
    num_results=5
)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [24]:
def search(question, course='llm-zoomcamp'):
    boost_dict={'question' : 2.0, 'section' : 0.5}
    filter_dict={'course' : 'llm-zoomcamp'}

    return index.search(
        question,
        boost_dict=boost_dict, 
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
# test - search from index
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

## Building a prompt

When we build AI systems, we usually split the prompt into two parts:

Instructions (also called the system prompt): this tells the LLM how to behave. It never changes, so it's the same for every request.
User prompt: this changes with every request. It carries the actual question and the retrieved context.

In [26]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [29]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

<h3>Building the context</h3>

Each document becomes a block with the section, question, and answer. This format makes it easy for the LLM to read. We turned a list of dictionaries into one string. It's a small preprocessing step before we send the data to the LLM. The 'context' is a formatted string with all the search results:

In [30]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [32]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [37]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context
    )
    return prompt.strip() # strip : remove all spaces ' '

In [ ]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [62]:
response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
print(response.model_dump_json(indent=2))

{
  "id": "resp_04d26feefd5451c3006a36f3690c14819788da35d5b394e98c",
  "created_at": 1781986153.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_04d26feefd5451c3006a36f369b20c8197910dd71b988dc898",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can still join now and start learning.\n\nIf you want to receive a certificate, make sure you submit your project while submissions are still open.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1781986153.0,
  "conversation": null,
  "max_output_tokens": null,
  "max

In [48]:
response.output[0].content[0].text

'Yes — you can join now and start learning.\n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions.'

In [49]:
response.usage

ResponseUsage(input_tokens=480, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=42, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=522)

In [50]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000549

In [ ]:
response.output_text

'Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.'

In [ ]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt},
    ]
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    return response.output_text

In [ ]:
def rag(question, model="gpt-5.4-mini"):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    return llm(INSTRUCTIONS, prompt, model=model)

In [61]:
answer = rag(question)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still open.


## RAG Helper | code into files

We'll use this code throughout the course, so let's put it into two reusable files:

ingest.py - loading data and building the search index
rag_helper.py - the RAG logic (search, prompt, LLM)